# Eksplorasi AutoModelForMultipleChoice

**Task**: Multiple Choice (Pilihan Ganda)
**Cara Kerja**: Model menerima satu konteks/pertanyaan, digabungkan dengan beberapa pilihan jawaban secara paralel. Model lalu memberikan skor/logits untuk masing-masing pilihan jawaban tersebut, dan yang mendapat probabilitas tertinggi akan dipilih sebagai jawaban yang benar.
**Model Populer**: BERT, RoBERTa, DeBERTa.
**Dataset**: `swag` (Situations With Adversarial Generations) - dataset populer yang memberikan sebuah konteks awal dari sebuah kejadian, lalu model harus menebak mana dari 4 pilihan kalimat ending yang paling masuk akal berkelanjutan.

In [1]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from datasets import load_dataset
import torch

## 1. Load Dataset Publik (`swag`)
Dataset SWAG secara spesifik memiliki data konteks (`sent1`), awal kalimat sambungan (`sent2`) dan 4 pilihan kelanjutan kalimat (`ending0` sampai `ending3`), lalu `label` mulai dari 0 hingga 3.

In [5]:
dataset = load_dataset("swag", "regular", split="train")

print("--- Contoh Data Index-0 ---")
print("Konteks awal (sent1) :", dataset[0]['sent1'])
print("Awal kalimat (sent2) :", dataset[0]['sent2'], "\n")

print("Pilihan 0:", dataset[0]['ending0'])
print("Pilihan 1:", dataset[0]['ending1'])
print("Pilihan 2:", dataset[0]['ending2'])
print("Pilihan 3:", dataset[0]['ending3'])

label = dataset[0]['label']
print("\nLabel Benar:", label, f"(Pilihan {label})")

--- Contoh Data Index-0 ---
Konteks awal (sent1) : Members of the procession walk down the street holding small horn brass instruments.
Awal kalimat (sent2) : A drum line 

Pilihan 0: passes by walking down the street playing their instruments.
Pilihan 1: has heard approaching them.
Pilihan 2: arrives and they're outside dancing and asleep.
Pilihan 3: turns the lead singer watches the performance.

Label Benar: 0 (Pilihan 0)


## 2. Load Tokenizer & Model
Kita akan menggunakan arsitektur `bert-base-uncased`. Jika memakai pre-trained bert standar tanpa fine-tuning MultipleChoice spesifik, prediksi awalnya akan berupa tebakan random atau tidak memiliki akurasi yang baik. Namun dari segi bentuk input/output struktur, sudah sesuai dengan skenario Multiple Choice.

In [6]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Wajib menggunakan AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
print(model)

BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

## 3. Inferensi Manual (Tokenisasi Khusus Multiple Choice)
Ini adalah bagian paling menriknya. Untuk Multiple Choice, kita sebenarnya membuat token **N** buah pasang kombinasi `[Pertanyaan + Pilihan n]` di mana **N** adalah jumlah opsi jawaban.

Format tensor yang harus masuk ke AutoModelForMultipleChoice bukanlah `(batch_size, seq_length)` melainkan `(batch_size, num_choices, seq_length)`.

In [9]:
prompt = "The boy is running towards the swing. He is about to"
choices = [
    "jump over it.",
    "sit on it and play.",
    "eat an apple.",
    "drive a car."
]

# Kita gandakan prompt agar jumlahnya sama dengan jumlah opsi jawaban (ada 4 opsi = prompt di copy 4 kali)
prompts = [prompt] * len(choices)
print("Prompts:", prompts)

# Kemudian kita pasangkan/tokensisasi kombinasi Pertanyaan dan ke-4 opsi jawaban.
inputs = tokenizer(prompts, choices, return_tensors="pt", padding=True)
print("\nBentuk Awal input_ids tensor:", inputs["input_ids"].shape) # Akan menghasilkan ukuran [4 baris, jumlah_token]

# Karena model AutoModelForMultipleChoice meminta format khusus dengan 3 dimensi [batch=1, pilihan=4, sequence=...]
# Kita perlu menambah dimensi 'batch' di depan dengan fungsi unsqueeze(0)
inputs = {k: v.unsqueeze(0) for k, v in inputs.items()}
print("Bentuk tensor dimasukkan ke model:", inputs["input_ids"].shape)

with torch.no_grad():
    outputs = model(**inputs)
    
# Model mengembalikan 4 logits, mewakili skor setiap 'choice'
logits = outputs.logits
predicted_class_id = logits.argmax().item()

print("\n--- Hasil Prediksi ---")
print("Logits:", logits.numpy())
print("Opsi dengan Skor Tertinggi (Jawaban):", predicted_class_id)
print("Teks:", choices[predicted_class_id])

Prompts: ['The boy is running towards the swing. He is about to', 'The boy is running towards the swing. He is about to', 'The boy is running towards the swing. He is about to', 'The boy is running towards the swing. He is about to']

Bentuk Awal input_ids tensor: torch.Size([4, 21])
Bentuk tensor dimasukkan ke model: torch.Size([1, 4, 21])

--- Hasil Prediksi ---
Logits: [[-0.3359357  -0.42243433 -0.2123431  -0.3425082 ]]
Opsi dengan Skor Tertinggi (Jawaban): 2
Teks: eat an apple.


## 4. Persiapan Data untuk PyTorch Training Mandiri
Model `AutoModelForMultipleChoice` butuh penanganan *Dataset* dan *Tokenizer* yang sedikit kompleks. Data harus di-*flatten* (dijadikan 1 dimensi), di-tokenize, lalu di-*unflatten* kembali menjadi blok berisi bongkahan *(batch, jumlah_pilihan, token_length)*.

Kita akan memformat SWAG dataset agar terbagi menjadi dimensi-dimensi yang siap dicerna dataloader PyTorch.

In [10]:
from torch.utils.data import Dataset, DataLoader

# Ambil sampel subset kecil simulasi
train_sample = dataset.select(range(50))

ending_names = ["ending0", "ending1", "ending2", "ending3"]

def preprocess_function(examples):
    # Buat array duplikat context/sent1 sebanyak 4x per baris
    first_sentences = [[context] * 4 for context in examples["sent1"]]
    
    # Gabungkan awalan sent2 dengan masing-masing ending0 s.d ending3
    question_headers = examples["sent2"]
    second_sentences = [
        [f"{header} {examples[end][i]}" for end in ending_names] 
        for i, header in enumerate(question_headers)
    ]
    
    # Flatten menjadi 1D list panjang untuk mempermudah tokenisasi
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    # Tokenisasi pasangan (sent1, sent2+ending) secara simultan
    tokenized_examples = tokenizer(first_sentences, second_sentences, truncation=True, max_length=64, padding="max_length")
    
    # Un-flatten (Kembalikan menjadi bentuk kelompok 4 pilihan jawaban per 1 batch data)
    unflattened = {k: [tokenized_examples[k][i : i + 4] for i in range(0, len(tokenized_examples[k]), 4)] for k in tokenized_examples.keys()}
    return unflattened

tokenized_train = train_sample.map(preprocess_function, batched=True)

class MultipleChoiceDataset(Dataset):
    def __init__(self, token_data):
        self.data = token_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "labels": torch.tensor(item["label"]) # Target klasifikasi 0, 1, 2, atau 3
        }

train_dataloader = DataLoader(MultipleChoiceDataset(tokenized_train), batch_size=4, shuffle=True)
print(f"Total Batch: {len(train_dataloader)}")
print("Selesai menyiapkan PyTorch DataLoader untuk Multiple Choice!")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Total Batch: 13
Selesai menyiapkan PyTorch DataLoader untuk Multiple Choice!


In [11]:
for data in train_dataloader:
    print(data)
    break

{'input_ids': tensor([[[  101,  2619,  3030,  ...,     0,     0,     0],
         [  101,  2619,  3030,  ...,     0,     0,     0],
         [  101,  2619,  3030,  ...,     0,     0,     0],
         [  101,  2619,  3030,  ...,     0,     0,     0]],

        [[  101,  2619,  8011,  ...,     0,     0,     0],
         [  101,  2619,  8011,  ...,     0,     0,     0],
         [  101,  2619,  8011,  ...,     0,     0,     0],
         [  101,  2619,  8011,  ...,     0,     0,     0]],

        [[  101,  2066,  1037,  ...,     0,     0,     0],
         [  101,  2066,  1037,  ...,     0,     0,     0],
         [  101,  2066,  1037,  ...,     0,     0,     0],
         [  101,  2066,  1037,  ...,     0,     0,     0]],

        [[  101,  2619, 14020,  ...,     0,     0,     0],
         [  101,  2619, 14020,  ...,     0,     0,     0],
         [  101,  2619, 14020,  ...,     0,     0,     0],
         [  101,  2619, 14020,  ...,     0,     0,     0]]]), 'attention_mask': tensor([[[1, 1,

## 5. Proses PyTorch Training Loop
Karena _Multiple Choice_ pada dasarnya adalah masalah *Classification*, proses trainingnya mirip sekali dengan klasifikasi kategori, namun perbedaannya adalah tensor *input_ids*-nya akan melebar mencakup jumlah pilihan (Contoh: `4 pilihan opsional`). Parameter `labels` memuat angka eksak yang benar (misal dari opsi indeks 0 hingga 3).

In [12]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training ===")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        # Kirim matriks 3 Dimensi + Matriks label
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Logits berjumlah sama dengan rentang tensor (batch_size, 4), Loss adalah CrossEntropy
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

==== Memulai Training ===
Epoch 1 | Step 0 | Loss 1.3895
Epoch 1 | Step 5 | Loss 1.3783
Epoch 1 | Step 10 | Loss 1.3055
>> Rata-rata Train Loss Epoch 1: 1.3774



## 6. Evaluasi Kinerja (Akurasi Pilihan Ganda)
Sebagaimana lumrahnya problem tebak soal abcd, evaluasi metriknya tentu saja dengan menghitung ketepatan akurasi (Prediksi yang argmax-nya persis menebak Label Jawaban Benar dibagi rata dengan semua jumlah himpunan/baris soal.)

In [13]:
# Simulasikan data validation kecil dengan teknik prepare data sama
val_sample = dataset.select(range(50, 70))
tokenized_val = val_sample.map(preprocess_function, batched=True)
val_dataloader = DataLoader(MultipleChoiceDataset(tokenized_val), batch_size=4)

model.eval()
total_val_loss = 0
correct_predictions = 0
total_examples = 0

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_val_loss += outputs.loss.item()
        
        # Hitung kalkulasi jumlah prediksi akurat
        predictions = outputs.logits.argmax(dim=-1)
        correct_predictions += (predictions == labels).sum().item()
        total_examples += labels.size(0)

avg_val_loss = total_val_loss / len(val_dataloader)
accuracy = correct_predictions / total_examples

print(f"Validation Loss     : {avg_val_loss:.4f}")
print(f"Akurasi             : {accuracy:.4f} (atau {accuracy*100:.2f}%)")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Validation Loss     : 1.3838
Akurasi             : 0.3500 (atau 35.00%)


## 7. Inferensi Hasil Akhir (Post-Training)
Setelah di-train dengan soal-soal dan dipaksa merepresentasikan pilihan yang benar, model tidak lagi asal menebak. Logits yang keluar harusnya memfasilitasi pilihan logikal terbaik.

In [14]:
prompt = "The chef is sprinkling salt onto the steak. Then he"
choices = [
    "throws the steak into the garbage.",
    "places it gently on the hot grill.",
    "starts painting a picture.",
    "goes to sleep."
]

prompts = [prompt] * len(choices)
inputs = tokenizer(prompts, choices, return_tensors="pt", padding=True)
inputs = {k: v.unsqueeze(0).to(device) for k, v in inputs.items()} # Tambahkan .to(device) untuk transfer batch GPU/CPU

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    
predicted_class_id = outputs.logits.argmax().item()

print("--- Hasil Inferensi POST-TRAIN ---")
print("Konteks   :", prompt)
print(f"Jawaban   : [{predicted_class_id}] -> {choices[predicted_class_id]}")

--- Hasil Inferensi POST-TRAIN ---
Konteks   : The chef is sprinkling salt onto the steak. Then he
Jawaban   : [0] -> throws the steak into the garbage.
